# H&M 마지막 주 · 남은 학습/재사용 상태만 확인

GPU가 필요 없습니다. 학습 중인 Colab을 유지하고 별도 CPU 런타임에서 실행할 수 있습니다.
Drive의 protocol·완료 JSON·진행 JSON만 읽습니다. .pt 로드, 학습, test 재평가, 기존 파일 수정은 없습니다.
seed49·50의 M1/M3/M5-C를 실행폴더별로 표시합니다. 완료된 모형은 전체 저장지표를 바로 읽습니다.
이 표는 부분 진행 확인이며 최종 성공·실패 판정이나 설정 변경의 근거가 아닙니다.


In [ ]:
from pathlib import Path
from google.colab import drive
import subprocess,sys
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
SOURCE_COMMIT='09955c77dc60c78a16c05987adf0567b9982e885'
REPO=Path('/content/clv-hm-status-'+SOURCE_COMMIT[:12])
if not REPO.exists(): subprocess.run(['git','clone','-q','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','-q','origin'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q','--detach',SOURCE_COMMIT],check=True)
sys.path.insert(0,str(REPO))


In [ ]:
import pandas as pd
import clv_hm_final10_saved_status as audit
status=audit.inspect_saved()
display(pd.DataFrame(status['status']))
print(status['note'])
print('학습:',status['new_training'],'| test 재평가:',status['test_evaluations'])
print('확인한 실행:',status['protocols'])
completed=pd.DataFrame(status['completed_metrics'])
if not completed.empty:
    core=['run_hash','seed','model_id','recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50','price_purchase_amount_weighted_hit@10','vndcg@10']
    display(completed[[c for c in core if c in completed]])
    print('위 지표는 300epoch 완료모형의 저장값입니다. 최종 전체시드 판정은 아닙니다.')
else: print('이 실행 범위에서는 아직 300epoch 완료모형이 없습니다.')
